In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import json

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon


def load_corpus(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return pd.DataFrame(records)


CORPUS_PATH = "../../results/generated_corpus.jsonl"
df = load_corpus(CORPUS_PATH)
print(df["label"].value_counts())

label
stego    272
cover    272
Name: count, dtype: int64


In [2]:
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

ppl_tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
ppl_model = GPT2LMHeadModel.from_pretrained("gpt2")
ppl_model.eval()


def compute_perplexity(text, max_length=1024):
    input_ids = ppl_tokenizer(text, return_tensors="pt", truncation=True,
                              max_length=max_length).input_ids
    if input_ids.size(1) < 2:
        return float("nan")
    with torch.no_grad():
        loss = ppl_model(input_ids, labels=input_ids).loss
    return torch.exp(loss).item()


perplexities = []
for i, text in enumerate(df["text"]):
    try:
        perplexities.append(compute_perplexity(text))
    except Exception as e:
        print(f"[WARN] perplexity, row {i}: {e}")
        perplexities.append(float("nan"))
df["perplexity"] = perplexities

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


In [3]:
from sentence_transformers import SentenceTransformer, util

sbert = SentenceTransformer("all-MiniLM-L6-v2")

prompt_emb = sbert.encode(df["prompt"].tolist(), batch_size=32, show_progress_bar=False, convert_to_tensor=True)
text_emb = sbert.encode(df["text"].tolist(), batch_size=32, show_progress_bar=False, convert_to_tensor=True)

df["prompt_similarity"] = util.pairwise_cos_sim(prompt_emb, text_emb).cpu().numpy()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
import textstat

df["flesch_reading_ease"] = df["text"].apply(textstat.flesch_reading_ease)
df["flesch_kincaid_grade"] = df["text"].apply(textstat.flesch_kincaid_grade)

In [5]:
from transformers import pipeline

cola = pipeline("text-classification", model="ModelTC/bert-base-uncased-cola", device=-1)
print("Model label mapping:", cola.model.config.id2label)


def cola_score(text, max_length=128):
    try:
        result = cola(text, truncation=True, max_length=max_length)[0]
        label = result["label"].lower()
        if label.startswith("accept"):
            p = result["score"]
        elif label.startswith(("unaccept", "not_accept", "not-accept")):
            p = 1 - result["score"]
        elif label in ("label_1", "1"):
            p = result["score"]
        elif label in ("label_0", "0"):
            p = 1 - result["score"]
        else:
            p = result["score"]
        return p
    except Exception as e:
        print(f"[WARN] CoLA: {e}")
        return float("nan")


df["grammatical_acceptability"] = [cola_score(text) for text in df["text"]]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model label mapping: {0: 'unacceptable', 1: 'acceptable'}


In [6]:
stego_q = df[df["label"] == "stego"].set_index("prompt")
cover_q = df[df["label"] == "cover"].set_index("prompt").loc[stego_q.index]


def fmt(x):
    return f"{x:.3f}" if abs(x) < 1 else f"{x:.1f}"


def share(s, cond):
    return f"{100 * cond(s).mean():.1f}%"


def p_value(col):
    return f"{wilcoxon(stego_q[col].values, cover_q[col].values).pvalue:.2g}"


quality_table = pd.DataFrame([
    ["Perplexity - median / mean",
     f"{fmt(cover_q.perplexity.median())} / {fmt(cover_q.perplexity.mean())}",
     f"{fmt(stego_q.perplexity.median())} / {fmt(stego_q.perplexity.mean())}", p_value("perplexity")],
    ["Perplexity > 100 (share of texts)",
     share(cover_q.perplexity, lambda s: s > 100), share(stego_q.perplexity, lambda s: s > 100), "-"],
    ["Prompt similarity - mean",
     fmt(cover_q.prompt_similarity.mean()), fmt(stego_q.prompt_similarity.mean()), p_value("prompt_similarity")],
    ["Flesch Reading Ease - mean",
     fmt(cover_q.flesch_reading_ease.mean()), fmt(stego_q.flesch_reading_ease.mean()), p_value("flesch_reading_ease")],
    ["Flesch-Kincaid Grade - mean",
     fmt(cover_q.flesch_kincaid_grade.mean()), fmt(stego_q.flesch_kincaid_grade.mean()), p_value("flesch_kincaid_grade")],
    ["Grammatical acceptability - median / mean",
     f"{fmt(cover_q.grammatical_acceptability.median())} / {fmt(cover_q.grammatical_acceptability.mean())}",
     f"{fmt(stego_q.grammatical_acceptability.median())} / {fmt(stego_q.grammatical_acceptability.mean())}",
     p_value("grammatical_acceptability")],
    ["Acceptable texts (CoLA score > 0.5)",
     share(cover_q.grammatical_acceptability, lambda s: s > 0.5),
     share(stego_q.grammatical_acceptability, lambda s: s > 0.5), "-"],
], columns=["Metric", "Cover", "Stego", "p-value (Wilcoxon)"])
print(quality_table.to_string(index=False))

                                   Metric         Cover         Stego p-value (Wilcoxon)
               Perplexity - median / mean   39.7 / 44.2   47.0 / 74.4            1.8e-06
        Perplexity > 100 (share of texts)          2.6%         15.4%                  -
                 Prompt similarity - mean         0.550         0.577             0.0041
               Flesch Reading Ease - mean          46.0          47.9               0.17
              Flesch-Kincaid Grade - mean          12.0          11.7               0.23
Grammatical acceptability - median / mean 0.996 / 0.989 0.994 / 0.857            9.6e-15
      Acceptable texts (CoLA score > 0.5)         99.3%         86.0%                  -


In [7]:
gpt_score = pd.to_numeric(stego_q["score"])
print(f"GPT-4.1 score of stego texts: mean = {gpt_score.mean():.2f}, median = {gpt_score.median():.0f}")
print(f"Score >= 80: {100 * (gpt_score >= 80).mean():.1f}%")
print(f"Score 60-79: {100 * ((gpt_score >= 60) & (gpt_score < 80)).mean():.1f}%")
print(f"Score < 40: {100 * (gpt_score < 40).mean():.1f}%")

GPT-4.1 score of stego texts: mean = 73.01, median = 78
Score >= 80: 48.2%
Score 60-79: 32.0%
Score < 40: 9.9%
